# LoRA rank study: run everything

Works in **Colab** or **locally**.

- **Colab:** Runtime > Change runtime type > **A100** (or L4). Results are kept on Google Drive, so if the
  session disconnects, re-run all cells and the study resumes where it stopped.
- **Local:** use a kernel with the package installed (`pip install -e ".[dev]"`). Outputs go to
  `data/`, `runs/` and `results/` in the repo. `GPU` in the setup cell picks the device.

Every step skips work that already exists, so re-running is always safe.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, sys, pathlib
IN_COLAB = 'google.colab' in sys.modules
PY = sys.executable          # run every step with this kernel's python
GPU = '1'                    # local only: which GPU to use

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    STORE = '/content/drive/MyDrive/lora-code-study'   # runs and results persist here
else:
    os.environ.setdefault('CUDA_VISIBLE_DEVICES', GPU)
    ROOT = pathlib.Path.cwd().resolve()
    while not (ROOT / 'pyproject.toml').exists():
        ROOT = ROOT.parent
    os.chdir(ROOT)
    print('repo:', ROOT, '| python:', PY, '| CUDA_VISIBLE_DEVICES:', os.environ['CUDA_VISIBLE_DEVICES'])

In [ ]:
if IN_COLAB:
    REPO = 'https://github.com/SummmerKing/Lora-optimiser.git'
    !test -d /content/Lora-optimiser || git clone $REPO /content/Lora-optimiser
    %cd /content/Lora-optimiser
    !git pull -q
    !pip install -q -e ".[dev]"
    # Keep outputs on Drive via symlinks.
    !mkdir -p $STORE/runs $STORE/results $STORE/data
    !rm -rf runs results data && ln -s $STORE/runs runs && ln -s $STORE/results results && ln -s $STORE/data data
else:
    # local: install once with  pip install -e ".[dev]"
    import lora_code
    print('lora_code from', pathlib.Path(lora_code.__file__).parent)

## 1. Tests
CPU only, about 20 s. Covers masking, LoRA gradients, merging, the stats and the execution sandbox.

In [ ]:
!{PY} -m pytest -q

## 2. Data: decontaminate and tokenize

In [ ]:
!{PY} scripts/prepare_data.py

## 3. Learning-rate sweep (selected by validation loss only)

In [ ]:
!{PY} scripts/sweep_lr.py

## 4. Base model baseline

In [ ]:
!test -f results/base/summary.json || {PY} scripts/evaluate.py --base

## 5. Train and evaluate 4 configurations x 3 seeds

In [ ]:
import subprocess, pathlib
for cfg in ['lora_r4', 'lora_r16', 'lora_r64', 'loraplus_r16']:
    for seed in [0, 1, 2]:
        run = pathlib.Path(f'runs/{cfg}/seed{seed}')
        if not (run / 'adapter.pt').exists():
            subprocess.run([PY, 'scripts/train.py', '--config', cfg, '--seed', str(seed)], check=True)
        if not pathlib.Path(f'results/{cfg}/seed{seed}/summary.json').exists():
            subprocess.run([PY, 'scripts/evaluate.py', '--run', str(run)], check=True)

## 6. Aggregate

In [ ]:
!{PY} scripts/aggregate.py
from IPython.display import Image, display, Markdown
display(Markdown(open('results/summary.md').read()))
display(Image('results/figures/gain_over_base.png'))
display(Image('results/figures/val_loss.png'))

## 7. Some failures
A few HumanEval problems `lora_r16` got wrong, to see what the failures actually look like.

In [ ]:
import json, random
rows = [json.loads(l) for l in open('results/lora_r16/seed0/humaneval.jsonl')]
fails = [r for r in rows if not r['passed']]
print(f"{len(fails)} failures")
for r in random.Random(0).sample(fails, min(3, len(fails))):
    print('=' * 80, r['task_id'], r['status'])
    print(r['response'][:1200])
    print('--- stderr:', r['stderr_tail'][-300:])